In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import os
import re
import torch
import time
import gc
import pandas as pd
import random
import numpy as np
import json
import pickle
from tqdm import tqdm
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from langchain_core.documents import Document
# from langchain_text_splitters import RecursiveCharacterTextSplitter

ModuleNotFoundError: No module named 'langchain_text_splitters'

# load chunks and embeddings

In [ ]:
# ===================== CONFIG =====================
save_folder = "/content/drive/MyDrive/Book_Embeddings_100/jinaa5_nano"
model_name  = "jinaai/jina-embeddings-v5-text-nano-retrieval"

# ===================== LOAD MODEL =====================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading model on {device}...")

model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
if torch.cuda.is_available():
    model = model.half()
print("Model loaded!\n")

# ===================== RELOAD ALL SAVED EMBEDDINGS =====================
all_chunks_text = []
all_embeddings  = []
all_metadata    = []

pkl_files = [f for f in os.listdir(save_folder) if f.endswith('.pkl')]
print(f"Found {len(pkl_files)} embedding files")

for pkl_file in tqdm(pkl_files, desc="Loading embeddings"):
    file_path = os.path.join(save_folder, pkl_file)
    with open(file_path, "rb") as f:
        data = pickle.load(f)

    chunks     = data["chunks"]
    embeddings = data["embeddings"]
    book_title = data["book_title"]
    file_name  = data["file_name"]

    for i, (chunk, emb) in enumerate(zip(chunks, embeddings)):
        all_chunks_text.append(chunk)
        all_embeddings.append(emb)
        all_metadata.append({
            "file_name": file_name,
            "book_title": book_title,
            "chunk_id": i
        })

all_embeddings = np.array(all_embeddings).astype('float32')

print(f"""
============== LOADED DATA (jina-v5-nano) ==============
  Total books:      {len(pkl_files)}
  Total chunks:     {len(all_chunks_text)}
  Embedding dim:    {all_embeddings.shape[1]}
  Matrix size:      {all_embeddings.nbytes/1024**2:.2f} MB
=========================================================
""")

Loading model on cuda...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/274 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.22k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/312 [00:00<?, ?B/s]

Model loaded!

Found 100 embedding files


Loading embeddings: 100%|██████████| 100/100 [00:09<00:00, 10.12it/s]



============== LOADED DATA (jina-v5-nano) ==============
  Total books:      100
  Total chunks:     88275
  Embedding dim:    768
  Matrix size:      258.62 MB



# Load model for query encoding

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

model_name = "jinaai/jina-embeddings-v5-text-nano-retrieval"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
if torch.cuda.is_available():
    model = model.half()
print("Model loaded for query encoding!")

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Loading weights:   0%|          | 0/110 [00:00<?, ?it/s]

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'factor'}


Model loaded for query encoding!


# Build Qdrant + Weaviate indexes for jina-v5-nano

In [4]:
!pip install -q qdrant-client weaviate-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 652.7/652.7 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 3.8 MB/s eta 0:00:00


## Weaviate

In [ ]:
import weaviate
from weaviate.classes.config import Configure, Property, DataType

# ===================== RECREATE WEAVIATE COLLECTION WITH chunk_id =====================
# ---- Weaviate ----
client_weaviate = weaviate.connect_to_embedded()
weaviate_collection = "BooksJina5nano"

if client_weaviate.collections.exists(weaviate_collection):
    client_weaviate.collections.delete(weaviate_collection)

client_weaviate.collections.create(
    name=weaviate_collection,
    vectorizer_config=Configure.Vectorizer.none(),
    properties=[
        Property(name="file_name", data_type=DataType.TEXT),
        Property(name="chunk_id",  data_type=DataType.INT)
    ]
)

print("Re-indexing into Weaviate...")
weaviate_index_start = time.time()
col = client_weaviate.collections.get(weaviate_collection)
with col.batch.fixed_size(batch_size=512) as batch:
    for i in tqdm(range(len(all_embeddings)), desc="Weaviate upload"):
        batch.add_object(
            properties={
                "file_name": all_metadata[i]["file_name"],
                "chunk_id":  all_metadata[i]["chunk_id"]
            },
            vector=all_embeddings[i].tolist()
        )
weaviate_index_time = time.time() - weaviate_index_start
print(f"Weaviate indexing time: {weaviate_index_time:.2f}s\n")

# ---- Search functions ----
def qdrant_search_detailed(query_emb, k=50):
    results = client_qdrant.query_points(
        collection_name=qdrant_collection, query=query_emb, limit=k, with_payload=True
    ).points
    return [(r.payload["file_name"], r.payload.get("chunk_id")) for r in results]

def weaviate_search_detailed(query_emb, k=50):
    results = col.query.near_vector(near_vector=query_emb, limit=k, return_properties=["file_name", "chunk_id"])
    return [(r.properties["file_name"], r.properties.get("chunk_id")) for r in results.objects]

Re-indexing into Weaviate...


Weaviate upload: 100%|██████████| 88275/88275 [02:11<00:00, 671.82it/s]


Weaviate indexing time: 133.83s



## qdrant

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

dim = all_embeddings.shape[1]

# ---- Qdrant ----
client_qdrant = QdrantClient(":memory:")
qdrant_collection = "books_jina5nano"

client_qdrant.create_collection(
    collection_name=qdrant_collection,
    vectors_config=VectorParams(size=dim, distance=Distance.COSINE)
)

print("Re-indexing into Qdrant...")
qdrant_index_start = time.time()
for i in tqdm(range(0, len(all_embeddings), 512), desc="Qdrant upload"):
    batch_end = min(i + 512, len(all_embeddings))
    points = [
        PointStruct(
            id=j,
            vector=all_embeddings[j].tolist(),
            payload={
                "file_name": all_metadata[j]["file_name"],
                "chunk_id": all_metadata[j]["chunk_id"]
            }
        )
        for j in range(i, batch_end)
    ]
    client_qdrant.upsert(collection_name=qdrant_collection, points=points)
qdrant_index_time = time.time() - qdrant_index_start
print(f"Qdrant indexing time: {qdrant_index_time:.2f}s\n")

Re-indexing into Qdrant...


Qdrant upload:  23%|██▎       | 39/173 [00:14<01:07,  1.98it/s]/tmp/ipykernel_1220/2886232655.py:30: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20480 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client_qdrant.upsert(collection_name=qdrant_collection, points=points)
Qdrant upload: 100%|██████████| 173/173 [01:09<00:00,  2.50it/s]

Qdrant indexing time: 69.18s



# Evaluation fuction

In [10]:
# ===================== UPDATED SEARCH FUNCTIONS (return chunk-level info) =====================

def qdrant_search_detailed(query_emb, k=50):
    """Returns raw chunk-level results: list of (file_name, chunk_id) tuples in rank order"""
    results = client_qdrant.query_points(
        collection_name=qdrant_collection,
        query=query_emb,
        limit=k,
        with_payload=True
    ).points

    return [(r.payload["file_name"], r.payload.get("chunk_id")) for r in results]


def weaviate_search_detailed(query_emb, k=50):
    """Returns raw chunk-level results: list of (file_name, chunk_id) tuples in rank order"""
    results = col.query.near_vector(
        near_vector=query_emb,
        limit=k,
        return_properties=["file_name", "chunk_id"]
    )

    return [(r.properties["file_name"], r.properties.get("chunk_id")) for r in results.objects]

In [11]:
def evaluate_robustness_full(search_fn_detailed, name, benchmark_df, deletion_pct):
    doc_hits_at_1, doc_hits_at_5, doc_hits_at_10 = 0, 0, 0
    chunk_hits_at_1, chunk_hits_at_5, chunk_hits_at_10 = 0, 0, 0
    latencies = []

    for _, row in tqdm(benchmark_df.iterrows(), desc=f"Evaluating {name} ({deletion_pct}%)", total=len(benchmark_df)):
        query = row["noisy_query"]
        expected_file  = row["expected_file"]
        expected_chunk = row["expected_chunk_id"]

        query_emb = model.encode([query], normalize_embeddings=True)[0].tolist()

        t0 = time.time()
        raw_results = search_fn_detailed(query_emb, k=50)
        latency_ms = (time.time() - t0) * 1000
        latencies.append(latency_ms)

        # Chunk-level
        chunk_matches = [
            (fname == expected_file and cid == expected_chunk)
            for fname, cid in raw_results
        ]
        if any(chunk_matches[:1]):
            chunk_hits_at_1 += 1
        if any(chunk_matches[:5]):
            chunk_hits_at_5 += 1
        if any(chunk_matches[:10]):
            chunk_hits_at_10 += 1

        # Doc-level
        retrieved_files, seen = [], set()
        for fname, _ in raw_results:
            if fname not in seen:
                seen.add(fname)
                retrieved_files.append(fname)
            if len(retrieved_files) == 10:
                break

        if retrieved_files and retrieved_files[0] == expected_file:
            doc_hits_at_1 += 1
        if expected_file in retrieved_files[:5]:
            doc_hits_at_5 += 1
        if expected_file in retrieved_files[:10]:
            doc_hits_at_10 += 1

    n = len(benchmark_df)
    lat = np.array(latencies)

    return {
        "vector_store": name,
        "model": "jina-v5-nano",
        "test_type": f"{deletion_pct}%_token_deletion",
        "doc_hit@1":  round(doc_hits_at_1 / n, 4),
        "doc_hit@5":  round(doc_hits_at_5 / n, 4),
        "doc_hit@10": round(doc_hits_at_10 / n, 4),
        "chunk_hit@1":  round(chunk_hits_at_1 / n, 4),
        "chunk_hit@5":  round(chunk_hits_at_5 / n, 4),
        "chunk_hit@10": round(chunk_hits_at_10 / n, 4),
        "latency_mean_ms": round(lat.mean(), 2),
        "latency_p95_ms":  round(np.percentile(lat, 95), 2),
    }


# Create test benchmark with n% deletion rate

In [15]:
def create_robustness_benchmark(all_chunks_text, all_metadata, all_embeddings,
                                n_samples=1000, deletion_ratio=0.40, random_seed=42):
    """
    Create a robustness benchmark dataset with random token deletion.

    Parameters:
    -----------
    all_chunks_text : list
        List of all chunk texts
    all_metadata : list
        List of metadata dictionaries for each chunk
    all_embeddings : numpy array
        Array of embeddings for each chunk
    n_samples : int
        Number of samples to include in the benchmark (default: 1000)
    deletion_ratio : float
        Proportion of tokens to delete (default: 0.40)
    random_seed : int
        Random seed for reproducibility (default: 42)

    Returns:
    --------
    pandas.DataFrame
        DataFrame containing the benchmark dataset
    """
    import random
    import numpy as np
    import pandas as pd

    # Set random seeds
    random.seed(random_seed)
    np.random.seed(random_seed)

    # Sample random chunks
    total_chunks = len(all_chunks_text)
    sample_indices = random.sample(range(total_chunks), min(n_samples, total_chunks))

    print(f"Sampled {len(sample_indices)} chunks out of {total_chunks} total")

    sampled_chunks = [all_chunks_text[i] for i in sample_indices]
    sampled_metadata = [all_metadata[i] for i in sample_indices]
    sampled_embeddings = all_embeddings[sample_indices]

    # Delete random tokens
    def delete_random_tokens(text, ratio):
        tokens = text.split()
        n_delete = int(len(tokens) * ratio)

        if n_delete == 0 or len(tokens) <= 1:
            return text

        delete_indices = set(random.sample(range(len(tokens)), n_delete))
        kept_tokens = [t for i, t in enumerate(tokens) if i not in delete_indices]
        return " ".join(kept_tokens)

    noisy_queries = [delete_random_tokens(chunk, deletion_ratio) for chunk in sampled_chunks]

    print("\n--- Example corruption ---")
    print(f"Original: {sampled_chunks[0][:200]}...")
    print(f"Noisy:    {noisy_queries[0][:200]}...")

    # Build benchmark DataFrame
    robustness_benchmark_df = pd.DataFrame({
        "id": range(len(sample_indices)),
        "original_chunk": sampled_chunks,
        "noisy_query": noisy_queries,
        "expected_file": [m["file_name"] for m in sampled_metadata],
        "expected_chunk_id": [m["chunk_id"] for m in sampled_metadata],
        "original_chunk_index": sample_indices
    })

    print(f"\nBuilt robustness benchmark with {len(robustness_benchmark_df)} queries")

    return robustness_benchmark_df

# ===================== USAGE EXAMPLES =====================

# Example 1: 40% deletion (default)
benchmark_df_40 = create_robustness_benchmark(
    all_chunks_text, all_metadata, all_embeddings,
    n_samples=1000,
    deletion_ratio=0.40,  # <-- Change this value
    random_seed=42
)
benchmark_df_40.to_csv("/content/drive/MyDrive/robustness_benchmark_1000_jina5nano_40pct.csv", index=False)

# Example 2: 30% deletion - just change the ratio!
benchmark_df_30 = create_robustness_benchmark(
    all_chunks_text, all_metadata, all_embeddings,
    n_samples=1000,
    deletion_ratio=0.30,  # <-- Change this value
    random_seed=42
)
benchmark_df_30.to_csv("/content/drive/MyDrive/robustness_benchmark_1000_jina5nano_30pct.csv", index=False)



Sampled 1000 chunks out of 88275 total

--- Example corruption ---
Original: پیامبری که وجه و خلیفه تام خداوند بر روی زمین بوده و واسطه و رحمت و فیض عوالم وجوی است بهفردی جز کسی که شایستگی این خطابات را داشته باشد، این کلام را نمی فرماید. عبارت دیگر قد اذنت لک است که اشاره دار...
Noisy:    که وجه و تام روی زمین بوده واسطه و و عوالم وجوی است جز کسی که این را داشته باشد، این نمی عبارت قد اذنت لک دارد این قبل از این این شما داده ام. ... نحو الكساء و السلام أبتاه يا أ تأذن لي أكون معكم الكس...

Built robustness benchmark with 1000 queries
Sampled 1000 chunks out of 88275 total

--- Example corruption ---
Original: پیامبری که وجه و خلیفه تام خداوند بر روی زمین بوده و واسطه و رحمت و فیض عوالم وجوی است بهفردی جز کسی که شایستگی این خطابات را داشته باشد، این کلام را نمی فرماید. عبارت دیگر قد اذنت لک است که اشاره دار...
Noisy:    که وجه و تام بر روی زمین بوده واسطه و و فیض عوالم وجوی است جز کسی که این خطابات را داشته باشد، این کلام نمی فرماید. عبارت قد اذنت لک دارد به این قبل از این این شما داده

## Evaluate

# 10 percent

In [ ]:
benchmark_df_10 = create_robustness_benchmark(
    all_chunks_text, all_metadata, all_embeddings,
    n_samples=1000,
    deletion_ratio=0.10,  # <-- Change this value
    random_seed=42
)

benchmark_df_10.to_csv("/content/drive/MyDrive/robustness_benchmark_1000_jina5nano_10pct.csv", index=False)
qdrant_robustness_10   = evaluate_robustness_full(qdrant_search_detailed, "Qdrant", benchmark_df_10, 10)
weaviate_robustness_10 = evaluate_robustness_full(weaviate_search_detailed, "Weaviate", benchmark_df_10, 10)

robustness_10_df = pd.DataFrame([qdrant_robustness_10, weaviate_robustness_10])
print(robustness_10_df.to_string(index=False))

robustness_10_df.to_csv("/content/drive/MyDrive/robustness_results_jina5nano_10pct.csv", index=False)

Sampled 1000 chunks out of 88275 total

--- Example corruption ---
Original: پیامبری که وجه و خلیفه تام خداوند بر روی زمین بوده و واسطه و رحمت و فیض عوالم وجوی است بهفردی جز کسی که شایستگی این خطابات را داشته باشد، این کلام را نمی فرماید. عبارت دیگر قد اذنت لک است که اشاره دار...
Noisy:    پیامبری که وجه و خلیفه تام خداوند بر روی زمین بوده و واسطه و رحمت و فیض عوالم وجوی است جز کسی که شایستگی این خطابات را داشته باشد، این کلام را نمی فرماید. عبارت قد اذنت لک است که دارد به این که قبل از...

Built robustness benchmark with 1000 queries


Evaluating Qdrant (10%):   0%|          | 0/1000 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Evaluating Weaviate (10%): 100%|██████████| 1000/1000 [00:25<00:00, 38.64it/s]

vector_store        model          test_type  doc_hit@1  doc_hit@5  doc_hit@10  chunk_hit@1  chunk_hit@5  chunk_hit@10  latency_mean_ms  latency_p95_ms
      Qdrant jina-v5-nano 10%_token_deletion      0.857      0.980       0.983        0.844        0.967         0.967           383.82          470.22
    Weaviate jina-v5-nano 10%_token_deletion      0.859      0.977       0.979        0.847        0.964         0.965             8.77           11.49


# 40 percent

In [ ]:
# Example 3: 40% deletion - super easy to change!
benchmark_df_40 = create_robustness_benchmark(
    all_chunks_text, all_metadata, all_embeddings,
    n_samples=1000,
    deletion_ratio=0.40,  # <-- Change this value
    random_seed=42
)
benchmark_df_40.to_csv("/content/drive/MyDrive/robustness_benchmark_1000_jina5nano_40pct.csv", index=False)

Sampled 1000 chunks out of 88275 total

--- Example corruption ---
Original: پیامبری که وجه و خلیفه تام خداوند بر روی زمین بوده و واسطه و رحمت و فیض عوالم وجوی است بهفردی جز کسی که شایستگی این خطابات را داشته باشد، این کلام را نمی فرماید. عبارت دیگر قد اذنت لک است که اشاره دار...
Noisy:    که وجه و تام روی زمین بوده واسطه و و عوالم وجوی است جز کسی که این را داشته باشد، این نمی عبارت قد اذنت لک دارد این قبل از این این شما داده ام. ... نحو الكساء و السلام أبتاه يا أ تأذن لي أكون معكم الكس...

Built robustness benchmark with 1000 queries


In [ ]:

qdrant_robustness_40   = evaluate_robustness_full(qdrant_search_detailed, "Qdrant", benchmark_df_40, 40)
weaviate_robustness_40 = evaluate_robustness_full(weaviate_search_detailed, "Weaviate", benchmark_df_40, 40)

robustness_40_df = pd.DataFrame([qdrant_robustness_40, weaviate_robustness_40])
print(robustness_40_df.to_string(index=False))

robustness_40_df.to_csv("/content/drive/MyDrive/robustness_results_jina5nano_40pct.csv", index=False)

vector_store        model          test_type  doc_hit@1  doc_hit@5  doc_hit@10  chunk_hit@1  chunk_hit@5  chunk_hit@10  latency_mean_ms  latency_p95_ms
      Qdrant jina-v5-nano 40%_token_deletion      0.811      0.954       0.968        0.783        0.931         0.937           386.27          472.16
    Weaviate jina-v5-nano 40%_token_deletion      0.826      0.948       0.962        0.797        0.925         0.930             8.99           12.00


# 50 percent

In [ ]:
benchmark_df_50 = create_robustness_benchmark(
    all_chunks_text, all_metadata, all_embeddings,
    n_samples=1000,
    deletion_ratio=0.50,  # <-- Change this value
    random_seed=42
)
benchmark_df_50.to_csv("/content/drive/MyDrive/robustness_benchmark_1000_jina5nano_50pct.csv", index=False)

qdrant_robustness_50   = evaluate_robustness_full(qdrant_search_detailed, "Qdrant", benchmark_df_50, 50)
weaviate_robustness_50 = evaluate_robustness_full(weaviate_search_detailed, "Weaviate", benchmark_df_50, 50)

robustness_50_df = pd.DataFrame([qdrant_robustness_50, weaviate_robustness_50])
print(robustness_50_df.to_string(index=False))

robustness_50_df.to_csv("/content/drive/MyDrive/robustness_results_jina5nano_50pct.csv", index=False)


Sampled 1000 chunks out of 88275 total

--- Example corruption ---
Original: پیامبری که وجه و خلیفه تام خداوند بر روی زمین بوده و واسطه و رحمت و فیض عوالم وجوی است بهفردی جز کسی که شایستگی این خطابات را داشته باشد، این کلام را نمی فرماید. عبارت دیگر قد اذنت لک است که اشاره دار...
Noisy:    وجه و تام روی زمین واسطه و عوالم وجوی جز کسی که این را داشته باشد، این نمی قد لک دارد قبل از این این شما داده ام. ... نحو الكساء و السلام أبتاه يا أ تأذن لي أكون معكم قال عليك السلام يا و يا بضعتي ص: ...

Built robustness benchmark with 1000 queries


Evaluating Qdrant (50%):   0%|          | 0/1000 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Evaluating Weaviate (50%): 100%|██████████| 1000/1000 [00:26<00:00, 37.75it/s]


vector_store        model          test_type  doc_hit@1  doc_hit@5  doc_hit@10  chunk_hit@1  chunk_hit@5  chunk_hit@10  latency_mean_ms  latency_p95_ms
      Qdrant jina-v5-nano 50%_token_deletion      0.751      0.917       0.943        0.717        0.870         0.881           383.80          470.63
    Weaviate jina-v5-nano 50%_token_deletion      0.743      0.906       0.936        0.705        0.856         0.869             9.24           12.37


# jinaa3

In [2]:
!pip install -q \
  sentence-transformers==3.3.1 \
  transformers==4.47.1 \
  tokenizers==0.21.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 131.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 114.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.7 MB/s eta 0:00:00


# Load chunks and embeddings

In [2]:
# ===================== CONFIG =====================
save_folder = "/content/drive/MyDrive/Book_Embeddings_100/jinaa3"
model_name  = "jinaai/jina-embeddings-v3"

# ===================== LOAD MODEL =====================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading model on {device}...")

model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
if torch.cuda.is_available():
    model = model.half()
print("Model loaded!\n")

# ===================== RELOAD ALL SAVED EMBEDDINGS =====================
all_chunks_text = []
all_embeddings  = []
all_metadata    = []

pkl_files = [f for f in os.listdir(save_folder) if f.endswith('.pkl')]
print(f"Found {len(pkl_files)} embedding files")

for pkl_file in tqdm(pkl_files, desc="Loading embeddings"):
    file_path = os.path.join(save_folder, pkl_file)
    with open(file_path, "rb") as f:
        data = pickle.load(f)

    chunks     = data["chunks"]
    embeddings = data["embeddings"]
    book_title = data["book_title"]
    file_name  = data["file_name"]

    for i, (chunk, emb) in enumerate(zip(chunks, embeddings)):
        all_chunks_text.append(chunk)
        all_embeddings.append(emb)
        all_metadata.append({
            "file_name": file_name,
            "book_title": book_title,
            "chunk_id": i
        })

all_embeddings = np.array(all_embeddings).astype('float32')

print(f"""
============== LOADED DATA (jina-v5-nano) ==============
  Total books:      {len(pkl_files)}
  Total chunks:     {len(all_chunks_text)}
  Embedding dim:    {all_embeddings.shape[1]}
  Matrix size:      {all_embeddings.nbytes/1024**2:.2f} MB
=========================================================
""")

Loading model on cuda...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/378 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/464 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/192 [00:00<?, ?B/s]

Model loaded!

Found 100 embedding files


Loading embeddings: 100%|██████████| 100/100 [00:24<00:00,  4.03it/s]



============== LOADED DATA (jina-v5-nano) ==============
  Total books:      100
  Total chunks:     88275
  Embedding dim:    1024
  Matrix size:      344.82 MB



# Load model for query encoding

In [3]:
import torch
from sentence_transformers import SentenceTransformer

model_name = "jinaai/jina-embeddings-v3"
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
if torch.cuda.is_available():
    model = model.half()
print("Model loaded for query encoding!")

Model loaded for query encoding!


# Build Qdrant + Weaviate indexes for jina-v5-nano

In [5]:
import weaviate
from weaviate.classes.config import Configure, Property, DataType

# ===================== RECREATE WEAVIATE COLLECTION WITH chunk_id =====================
# ---- Weaviate ----
client_weaviate = weaviate.connect_to_embedded()
weaviate_collection = "BooksJina3"

if client_weaviate.collections.exists(weaviate_collection):
    client_weaviate.collections.delete(weaviate_collection)

client_weaviate.collections.create(
    name=weaviate_collection,
    vectorizer_config=Configure.Vectorizer.none(),
    properties=[
        Property(name="file_name", data_type=DataType.TEXT),
        Property(name="chunk_id",  data_type=DataType.INT)
    ]
)

print("Re-indexing into Weaviate...")
weaviate_index_start = time.time()
col = client_weaviate.collections.get(weaviate_collection)
with col.batch.fixed_size(batch_size=512) as batch:
    for i in tqdm(range(len(all_embeddings)), desc="Weaviate upload"):
        batch.add_object(
            properties={
                "file_name": all_metadata[i]["file_name"],
                "chunk_id":  all_metadata[i]["chunk_id"]
            },
            vector=all_embeddings[i].tolist()
        )
weaviate_index_time = time.time() - weaviate_index_start
print(f"Weaviate indexing time: {weaviate_index_time:.2f}s\n")

# ---- Search functions ----
def qdrant_search_detailed(query_emb, k=50):
    results = client_qdrant.query_points(
        collection_name=qdrant_collection, query=query_emb, limit=k, with_payload=True
    ).points
    return [(r.payload["file_name"], r.payload.get("chunk_id")) for r in results]

def weaviate_search_detailed(query_emb, k=50):
    results = col.query.near_vector(near_vector=query_emb, limit=k, return_properties=["file_name", "chunk_id"])
    return [(r.properties["file_name"], r.properties.get("chunk_id")) for r in results.objects]

INFO:weaviate-client:Binary /root/.cache/weaviate-embedded did not exist. Downloading binary from https://github.com/weaviate/weaviate/releases/download/v1.30.5/weaviate-v1.30.5-Linux-amd64.tar.gz
INFO:weaviate-client:Started /root/.cache/weaviate-embedded: process ID 3040


Re-indexing into Weaviate...


Weaviate upload: 100%|██████████| 88275/88275 [02:15<00:00, 653.82it/s]


Weaviate indexing time: 138.42s



## qdrant

In [6]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

dim = all_embeddings.shape[1]

# ---- Qdrant ----
client_qdrant = QdrantClient(":memory:")
qdrant_collection = "books_jina3"

client_qdrant.create_collection(
    collection_name=qdrant_collection,
    vectors_config=VectorParams(size=dim, distance=Distance.COSINE)
)

print("Re-indexing into Qdrant...")
qdrant_index_start = time.time()
for i in tqdm(range(0, len(all_embeddings), 512), desc="Qdrant upload"):
    batch_end = min(i + 512, len(all_embeddings))
    points = [
        PointStruct(
            id=j,
            vector=all_embeddings[j].tolist(),
            payload={
                "file_name": all_metadata[j]["file_name"],
                "chunk_id": all_metadata[j]["chunk_id"]
            }
        )
        for j in range(i, batch_end)
    ]
    client_qdrant.upsert(collection_name=qdrant_collection, points=points)
qdrant_index_time = time.time() - qdrant_index_start
print(f"Qdrant indexing time: {qdrant_index_time:.2f}s\n")

Re-indexing into Qdrant...


Qdrant upload:  23%|██▎       | 39/173 [00:17<00:56,  2.37it/s]/tmp/ipykernel_1558/961077417.py:30: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20480 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client_qdrant.upsert(collection_name=qdrant_collection, points=points)
Qdrant upload: 100%|██████████| 173/173 [01:27<00:00,  1.98it/s]

Qdrant indexing time: 87.22s



# Evaluation

# 10 percent

In [16]:
benchmark_df_10 = create_robustness_benchmark(
    all_chunks_text, all_metadata, all_embeddings,
    n_samples=1000,
    deletion_ratio=0.10,  # <-- Change this value
    random_seed=42
)
benchmark_df_10.to_csv("/content/drive/MyDrive/robustness_benchmark_1000_jina5nano_10pct.csv", index=False)


qdrant_robustness_10   = evaluate_robustness_full(qdrant_search_detailed, "Qdrant", benchmark_df_10, 10)
weaviate_robustness_10 = evaluate_robustness_full(weaviate_search_detailed, "Weaviate", benchmark_df_10, 10)

robustness_10_df = pd.DataFrame([qdrant_robustness_10, weaviate_robustness_10])
print(robustness_10_df.to_string(index=False))

robustness_10_df.to_csv("/content/drive/MyDrive/robustness_results_jina3_10pct.csv", index=False)

Sampled 1000 chunks out of 88275 total

--- Example corruption ---
Original: پیامبری که وجه و خلیفه تام خداوند بر روی زمین بوده و واسطه و رحمت و فیض عوالم وجوی است بهفردی جز کسی که شایستگی این خطابات را داشته باشد، این کلام را نمی فرماید. عبارت دیگر قد اذنت لک است که اشاره دار...
Noisy:    پیامبری که وجه و خلیفه تام خداوند بر روی زمین بوده و واسطه و رحمت و فیض عوالم وجوی است جز کسی که شایستگی این خطابات را داشته باشد، این کلام را نمی فرماید. عبارت قد اذنت لک است که دارد به این که قبل از...

Built robustness benchmark with 1000 queries


Evaluating Weaviate (10%): 100%|██████████| 1000/1000 [01:00<00:00, 16.44it/s]

vector_store        model          test_type  doc_hit@1  doc_hit@5  doc_hit@10  chunk_hit@1  chunk_hit@5  chunk_hit@10  latency_mean_ms  latency_p95_ms
      Qdrant jina-v5-nano 10%_token_deletion      0.876      0.984       0.986        0.861        0.980         0.981           499.34          577.52
    Weaviate jina-v5-nano 10%_token_deletion      0.877      0.983       0.986        0.866        0.978         0.980             9.30           12.25


# 20 percent

In [12]:
benchmark_df_20 = pd.read_csv("/content/drive/MyDrive/robustness_benchmark_20.csv")


qdrant_robustness_20   = evaluate_robustness_full(qdrant_search_detailed, "Qdrant", benchmark_df_20, 20)
weaviate_robustness_20 = evaluate_robustness_full(weaviate_search_detailed, "Weaviate", benchmark_df_20, 20)

robustness_20_df = pd.DataFrame([qdrant_robustness_20, weaviate_robustness_20])
print(robustness_20_df.to_string(index=False))

robustness_20_df.to_csv("/content/drive/MyDrive/robustness_results_jina3_20pct.csv", index=False)

Evaluating Weaviate (20%): 100%|██████████| 1000/1000 [01:01<00:00, 16.13it/s]

vector_store        model          test_type  doc_hit@1  doc_hit@5  doc_hit@10  chunk_hit@1  chunk_hit@5  chunk_hit@10  latency_mean_ms  latency_p95_ms
      Qdrant jina-v5-nano 20%_token_deletion      0.881      0.984       0.986        0.865        0.979         0.981           508.32          592.19
    Weaviate jina-v5-nano 20%_token_deletion      0.875      0.982       0.986        0.859        0.976         0.979             9.64           12.29


# 30 percent

In [13]:
benchmark_df_30 = pd.read_csv("/content/drive/MyDrive/robustness_benchmark_1000_jina5nano_30pct.csv")


qdrant_robustness_30   = evaluate_robustness_full(qdrant_search_detailed, "Qdrant", benchmark_df_30, 30)
weaviate_robustness_30 = evaluate_robustness_full(weaviate_search_detailed, "Weaviate", benchmark_df_30, 30)

robustness_30_df = pd.DataFrame([qdrant_robustness_30, weaviate_robustness_30])
print(robustness_30_df.to_string(index=False))

robustness_30_df.to_csv("/content/drive/MyDrive/robustness_results_jina3_30pct.csv", index=False)

Evaluating Weaviate (30%): 100%|██████████| 1000/1000 [01:02<00:00, 16.03it/s]

vector_store        model          test_type  doc_hit@1  doc_hit@5  doc_hit@10  chunk_hit@1  chunk_hit@5  chunk_hit@10  latency_mean_ms  latency_p95_ms
      Qdrant jina-v5-nano 30%_token_deletion      0.861      0.981       0.984        0.840        0.972         0.977           507.05          594.99
    Weaviate jina-v5-nano 30%_token_deletion      0.845      0.980       0.984        0.819        0.969         0.974             9.84           12.79


# 40 percent

In [ ]:
benchmark_df_40 = pd.read_csv("/content/drive/MyDrive/robustness_benchmark_1000_jina5nano_40pct.csv")


qdrant_robustness_40   = evaluate_robustness_full(qdrant_search_detailed, "Qdrant", benchmark_df_40, 40)
weaviate_robustness_40 = evaluate_robustness_full(weaviate_search_detailed, "Weaviate", benchmark_df_40, 40)

robustness_40_df = pd.DataFrame([qdrant_robustness_40, weaviate_robustness_40])
print(robustness_40_df.to_string(index=False))

robustness_40_df.to_csv("/content/drive/MyDrive/robustness_results_jina3_40pct.csv", index=False)

Evaluating Weaviate (40%): 100%|██████████| 1000/1000 [01:00<00:00, 16.54it/s]

vector_store        model          test_type  doc_hit@1  doc_hit@5  doc_hit@10  chunk_hit@1  chunk_hit@5  chunk_hit@10  latency_mean_ms  latency_p95_ms
      Qdrant jina-v5-nano 40%_token_deletion      0.857      0.977       0.984        0.827        0.958         0.964           492.30          579.19
    Weaviate jina-v5-nano 40%_token_deletion      0.845      0.975       0.980        0.818        0.955         0.960             9.39           12.48


# 50 percent

In [ ]:
benchmark_df_50 = pd.read_csv("/content/drive/MyDrive/robustness_benchmark_1000_jina5nano_50pct.csv")


qdrant_robustness_50   = evaluate_robustness_full(qdrant_search_detailed, "Qdrant", benchmark_df_50, 50)
weaviate_robustness_50 = evaluate_robustness_full(weaviate_search_detailed, "Weaviate", benchmark_df_50, 50)

robustness_50_df = pd.DataFrame([qdrant_robustness_50, weaviate_robustness_50])
print(robustness_50_df.to_string(index=False))

robustness_50_df.to_csv("/content/drive/MyDrive/robustness_results_jina3_50pct.csv", index=False)

Evaluating Weaviate (50%): 100%|██████████| 1000/1000 [01:01<00:00, 16.39it/s]

vector_store        model          test_type  doc_hit@1  doc_hit@5  doc_hit@10  chunk_hit@1  chunk_hit@5  chunk_hit@10  latency_mean_ms  latency_p95_ms
      Qdrant jina-v5-nano 50%_token_deletion      0.779      0.955       0.972        0.738        0.921         0.937           489.41          580.32
    Weaviate jina-v5-nano 50%_token_deletion      0.776      0.943       0.962        0.735        0.908         0.923             9.48           12.42
